<a href="https://colab.research.google.com/github/Fabimotter/AED2/blob/main/Aula6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercicio 5

In [3]:
def decimal_para_hexa(n):
    if not isinstance(n, int) or n < 0:
        raise ValueError("Digite um número inteiro não negativo.")

    if n == 0:
        return "0"

    pilha = []
    digitos = "0123456789ABCDEF"

    while n > 0:
        resto = n % 16
        pilha.append(digitos[resto])
        n = n // 16

    hexadecimal = ""
    while pilha:
        hexadecimal += pilha.pop()
    return hexadecimal


for decimal in [0, 15, 26, 100, 255, 4095]:
    print(f"Decimal: {decimal:>4} -> Hexadecimal: {decimal_para_hexa(decimal)}")

assert decimal_para_hexa(26) == "1A"
assert decimal_para_hexa(0) == "0"

Decimal:    0 -> Hexadecimal: 0
Decimal:   15 -> Hexadecimal: F
Decimal:   26 -> Hexadecimal: 1A
Decimal:  100 -> Hexadecimal: 64
Decimal:  255 -> Hexadecimal: FF
Decimal: 4095 -> Hexadecimal: FFF


Exercicio 6

In [4]:
def balanceia_colchetes(expressao):
    pilha = []
    pares = {')': '(', ']': '[', '}': '{'}

    for caractere in expressao:
        if caractere in "([{":
            pilha.append(caractere)
        elif caractere in ")]}":
            if not pilha:
                return False
            if pilha.pop() != pares[caractere]:
                return False

    return len(pilha) == 0


expressoes = ["({[]})", "({[}])", "(a+b)", "((a+b)", "a+b", "[x*{y+1}]"]
for expressao in expressoes:
    print(f"{expressao:12} -> {balanceia_colchetes(expressao)}")

assert balanceia_colchetes("({[]})") is True
assert balanceia_colchetes("({[}])") is False

({[]})       -> True
({[}])       -> False
(a+b)        -> True
((a+b)       -> False
a+b          -> True
[x*{y+1}]    -> True


Projeto 1

In [5]:
import pandas as pd
from IPython.display import display


class HistoricoFiltros:
    def __init__(self, dataframe):
        self.original = dataframe.copy(deep=True)
        self.atual = dataframe.copy(deep=True)
        self.historico = []  # pilha de estados anteriores

    def aplicar_filtro(self, coluna, operador, valor):
        if coluna not in self.atual.columns:
            raise KeyError(f"Coluna inexistente: {coluna}")

        operacoes = {
            '==': lambda serie: serie == valor,
            '!=': lambda serie: serie != valor,
            '>': lambda serie: serie > valor,
            '>=': lambda serie: serie >= valor,
            '<': lambda serie: serie < valor,
            '<=': lambda serie: serie <= valor,
        }
        if operador not in operacoes:
            raise ValueError(f"Operador inválido: {operador}")

        filtrado = self.atual.loc[operacoes[operador](self.atual[coluna])].copy()
        self.historico.append(self.atual.copy(deep=True))  # PUSH
        self.atual = filtrado

    def desfazer(self):
        if not self.historico:
            print("Não há filtros para desfazer.")
            return False
        self.atual = self.historico.pop()  # POP
        return True

    def mostrar(self, titulo="Tabela atual"):
        print(f"\n{titulo} — {len(self.atual)} registro(s)")
        display(self.atual.reset_index(drop=True))

Prjeto 2

In [6]:
import re


def separar_tokens(expressao):
    padrao = r"\d+(?:\.\d+)?|\.\d+|[()+\-*/]"
    tokens = []
    posicao = 0

    for encontrado in re.finditer(padrao, expressao):
        if expressao[posicao:encontrado.start()].strip():
            raise ValueError("A expressão contém caracteres inválidos.")
        tokens.append(encontrado.group())
        posicao = encontrado.end()

    if expressao[posicao:].strip():
        raise ValueError("A expressão contém caracteres inválidos.")
    return tokens


def calcular_expressao(expressao):
    numeros = []
    operadores = []
    prioridade = {'+': 1, '-': 1, '*': 2, '/': 2,
                  'u+': 3, 'u-': 3}

    def aplicar_operador():
        if not operadores:
            raise ValueError("Operador ausente.")
        operador = operadores.pop()

        if operador in ('u+', 'u-'):
            if not numeros:
                raise ValueError("Número ausente após sinal.")
            valor = numeros.pop()
            numeros.append(valor if operador == 'u+' else -valor)
            return

        if len(numeros) < 2:
            raise ValueError("Expressão inválida: faltam operandos.")
        b = numeros.pop()
        a = numeros.pop()

        if operador == '+':
            resultado = a + b
        elif operador == '-':
            resultado = a - b
        elif operador == '*':
            resultado = a * b
        elif operador == '/':
            if b == 0:
                raise ZeroDivisionError("Divisão por zero não é permitida.")
            resultado = a / b
        else:
            raise ValueError(f"Operador desconhecido: {operador}")

        numeros.append(resultado)

    esperando_numero = True
    for token in separar_tokens(expressao):
        if re.fullmatch(r"\d+(?:\.\d+)?|\.\d+", token):
            if not esperando_numero:
                raise ValueError("Falta operador entre os números.")
            numeros.append(float(token))
            esperando_numero = False

        elif token == '(':
            if not esperando_numero:
                raise ValueError("Falta operador antes de '('.")
            operadores.append(token)
            esperando_numero = True

        elif token == ')':
            if esperando_numero:
                raise ValueError("Parêntese de fechamento inesperado.")
            while operadores and operadores[-1] != '(':
                aplicar_operador()
            if not operadores:
                raise ValueError("Parênteses não balanceados.")
            operadores.pop()  # remove '('
            if operadores and operadores[-1] in ('u+', 'u-'):
                aplicar_operador()
            esperando_numero = False

        else:  # operadores +, -, *, /
            if esperando_numero:
                if token in ('+', '-'):
                    operadores.append('u' + token)
                    continue
                raise ValueError("Operador sem operando à esquerda.")

            while (operadores and operadores[-1] != '(' and
                   prioridade[operadores[-1]] >= prioridade[token]):
                aplicar_operador()
            operadores.append(token)
            esperando_numero = True

    if esperando_numero:
        raise ValueError("Expressão incompleta ou vazia.")

    while operadores:
        if operadores[-1] == '(':
            raise ValueError("Parênteses não balanceados.")
        aplicar_operador()

    if len(numeros) != 1:
        raise ValueError("Expressão inválida.")
    return numeros[0]